# Admissions in the MScFE 🎓🗞

In this project, you conducted an experiment to help WQU improve enrollment in the Applied Data Science Lab. But let's not forget about our Master of Science in Financial Engineering! For your assignment, you'll help the MScFE conduct a similar experiment. This will be a great opportunity to put your new EDA, ETL, and statistics skills into action.

Also, keep in mind that for many of these submissions, you'll be passing in dictionaries that will test different parts of your code.

Note: Replace ip_of_mongo_device with your actual MongoDB IP address.

In [ ]:
from pymongo import MongoClient
from pymongo.collection import Collection
from teaching_tools.ab_test.reset import Reset

r = Reset("192.129.228.2")
r.reset_database()

In [ ]:
# Import your libraries here
from pprint import PrettyPrinter

import pandas as pd
import plotly.express as px
from country_converter import CountryConverter
from pymongo import MongoClient
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import scipy
from pymongo import MongoClient
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.power import GofChisquarePower
from teaching_tools.ab_test.experiment import Experiment
from teaching_tools.ab_test.reset import Reset

In [ ]:
host = "192.129.228.2"

On your MongoDB server, there is a collection named "mscfe-applicants". Locate this collection, and assign it to the variable name mscfe_app.


Note: When using the MongoClient class, specify the host by passing host=host as the first argument. For example: MongoClient(host=host, port=27017)

In [ ]:
# Create `client`
client = MongoClient(host=host, port=27017)
# Create `db`
db = client["wqu-abtest"]
# Assign `"mscfe-applicants"` collection to `mscfe_app`
mscfe_app = db["mscfe-applicants"]
print("client:", type(client))
print("mscfe_app:", type(mscfe_app))

## Explore

Aggregate the applicants in mscfe_app by nationality, and then load your results into the DataFrame df_nationality. Your DataFrame should have two columns: "country_iso2" and "count".

In [ ]:
pp = PrettyPrinter(indent=2)
print("pp type:", type(pp))

In [ ]:
pp.pprint(client.list_database_names())

In [ ]:
result = mscfe_app.find_one({})
print("result type:", type(result))
pp.pprint(result)

Using the country_converter library, add two new columns to df_nationality. The first, "country_name", should contain the short name of the country in each row. The second, "country_iso3", should contain the three-letter abbreviation.

In [ ]:
# Aggregate applicants by nationality
result = mscfe_app.aggregate([
    {
        "$group": {
            "_id": "$countryISO2",
            "count": {"$sum": 1}
        }
    }
])

# Load result into DataFrame
df_nationality = (
    pd.DataFrame(result)
    .rename(columns={"_id": "country_iso2"})
    .sort_values("country_iso2", ascending=True)
    .reset_index(drop=True)  # ✅ This line ensures index matches expected format
)

# Print checks
print("df_nationality type:", type(df_nationality))
print("df_nationality shape", df_nationality.shape)
df_nationality.head()


Build a function build_nat_choropleth that uses plotly express and the data in df_nationality to create a choropleth map of the nationalities of MScFE applicants. Be sure to use the title "MScFE Applicants: Nationalities".

In [ ]:
# Instantiate `CountryConverter`
cc = CountryConverter()

# Create `"country_name"` column
df_nationality["country_name"] = cc.convert(
    df_nationality["country_iso2"], to="name_short"
    
)

# Create `"country_iso3"` column
df_nationality["country_iso3"] = cc.convert(df_nationality["country_iso2"], to="ISO3")

print("df_nationality type:", type(df_nationality))
print("df_nationality shape", df_nationality.shape)
df_nationality.head()

In [ ]:
# Create `build_nat_choropleth` function
def build_nat_choropleth():
    fig = px.choropleth(
        data_frame=df_nationality,
        locations="country_iso3",
        color="count",
        projection="natural earth",
        color_continuous_scale=px.colors.sequential.Oranges,
        title="MScFE Applicants: Nationalities"
        
    )
    return fig



# Don't delete the code below 👇
nat_fig = build_nat_choropleth()
nat_fig.show()

In [ ]:
import random
import pandas as pd
from pymongo import MongoClient

class MongoRepository:
    """Repository class for interacting with MongoDB database."""
    
    def __init__(
        self,
        client=MongoClient(host=host, port=27017),
        db="wqu-abtest",
        collection="mscfe-applicants"
    ):
        self.collection = client[db][collection]

    def find_by_date(self, date_string):
        start = pd.to_datetime(date_string, format="%Y-%m-%d")
        end = start + pd.DateOffset(days=1)
        query = {"createdAt": {"$gte": start, "$lt": end}, "admissionsQuiz": "incomplete"}
        result = self.collection.find(query)
        return list(result)

    def update_applicants(self, observations):
        n = 0
        n_modified = 0
        for doc in observations:
            result = self.collection.update_one(
                filter={"_id": doc["_id"]},
                update={"$set": doc}
            )
            n += result.matched_count
            n_modified += result.modified_count
        return {"n": n, "nModified": n_modified}

    def assign_to_groups(self, date_string):
        observations = self.find_by_date(date_string)
        random.seed(42)
        random.shuffle(observations)

        idx = len(observations) // 2

        for doc in observations[:idx]:
            doc["inExperiment"] = True
            doc["group"] = "no email (control)"

        for doc in observations[idx:]:
            doc["inExperiment"] = True
            doc["group"] = "email (treatment)"

        return self.update_applicants(observations)

    def find_exp_observations(self):
        """Return all documents that were part of the experiment."""
        query = {"inExperiment": True}
        result = self.collection.find(query)
        return list(result)

In [ ]:
repo = MongoRepository()
print("repo type:", type(repo))
repo

In [ ]:
# Don't modify the code below, it will help test `find_by_date` method.
submission = repo.find_by_date("2022-06-01")
submission

In [ ]:
# Don't modify the code below, it will help test `assign_to_groups` method.
date = "2022-06-02"
submission = repo.assign_to_groups(date)

In [ ]:
chi_square_power = GofChisquarePower()
group_size = math.ceil(
    chi_square_power.solve_power(effect_size=0.5, alpha=0.05, power=0.8)
)

print("Group size:", group_size)
print("Total # of applicants needed:", group_size * 2)

In [ ]:
# Aggregate no-quiz applicants by sign-up date
result = mscfe_app.aggregate(
    [
        {"$match": {"admissionsQuiz": "incomplete"}},
        {
            "$group": {
                "_id": {"$dateTrunc": {"date":"$createdAt", "unit": "day"}},
                "count": {"$sum": 1}
            }
        }
    ]
)

print("result type:", type(result))

# Load result into DataFrame
no_quiz_mscfe = (
    pd.DataFrame(result)
    .rename({"_id": "date", "count": "new_users"}, axis = 1)
    .set_index("date")
    .sort_index()
    .squeeze()

)

print("no_quiz type:", type(no_quiz_mscfe))
print("no_quiz shape:", no_quiz_mscfe.shape)
no_quiz_mscfe.head()

In [ ]:
mean = no_quiz_mscfe.describe()["mean"]
std = no_quiz_mscfe.describe()["std"]
print("no_quiz mean:", mean)
print("no_quiz std:", std)

In [ ]:
exp_days = 7
sum_mean = mean * exp_days
sum_std = std * np.sqrt(exp_days)
print("Mean of sum:", sum_mean)
print("Std of sum:", sum_std)

In [ ]:
prob_65_or_fewer = scipy.stats.norm.cdf(
    group_size * 2,
    loc= sum_mean,
    scale=sum_std
)
prob_65_or_greater = 1- prob_65_or_fewer

print(
    f"Probability of getting 65+ no_quiz in {exp_days} days:",
    round(prob_65_or_greater, 3),
)

In [ ]:
exp = Experiment(repo=client, db="wqu-abtest", collection="mscfe-applicants")
exp.reset_experiment()
result = exp.run_experiment(days=exp_days, assignment=True)
print("result type:", type(result))
result

In [ ]:
# Don't modify the code below, it will help test `find_exp_observations` method
submission = repo.find_exp_observations()

In [ ]:
result = repo.find_exp_observations()
df = pd.DataFrame(result)

print("df type:", type(df))
print("df shape:", df.shape)
df.head()

In [ ]:
data = pd.crosstab(
    index=df["group"],
    columns=df["admissionsQuiz"],
    normalize=False
)

print("data type:", type(data))
print("data shape:", data.shape)
data

In [ ]:
# Create `build_contingency_bar` function
def build_contingency_bar():
    # Create side-by-side bar chart
    fig = px.bar(
        data_frame=data,
        barmode="group",
        title="MScFE: Admissions Quiz Completion by Group"
    )
    # Set axis labels
    fig.update_layout(
        xaxis_title="Group",
        yaxis_title="Frequency [count]",
        legend={"title": "Admissions Quiz"}
    )

    return fig

build_contingency_bar().show()


# Don't delete the code below 👇
cb_fig = build_contingency_bar()
cb_fig.show()

In [ ]:
contingency_table = Table2x2(data.values)

print("contingency_table type:", type(contingency_table))
contingency_table.table_orig

In [ ]:
chi_square_test = contingency_table.test_nominal_association()

print("chi_square_test type:", type(chi_square_test))
print(chi_square_test)

In [ ]:
odds_ratio = contingency_table.oddsratio.round(1)
print("Odds ratio:", odds_ratio)